<a href="https://colab.research.google.com/github/Jenn2626/Project-The-Look-eCommerce-Analysis-by-SQL/blob/main/Ad-hoc%20Question%20and%20Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ad-hoc Questions
---
In this analysis, we will be focusing on orders that were Completed

## 1. How much are we selling monthly?

### a. Numbers of orders and customers






In [ ]:
%%sql

WITH year_month_order AS
(
SELECT
EXTRACT(YEAR FROM created_at) AS ORDER_YEAR,
EXTRACT(MONTH FROM created_at) AS ORDER_MONTH,
COUNT( DISTINCT USER_ID) AS TOTAL_USERS,
COUNT( DISTINCTORDER_ID) AS TOTAL_ORDERS
FROM orders
WHERE STATUS IN('Complete')
GROUP BY ORDER_YEAR,ORDER_MONTH)

SELECT
CAST(ORDER_YEAR AS STRING) || '-' || CAST(ORDER_MONTH AS STRING) AS YEAR_MONTH,
TOTAL_USERS, TOTAL_ORDERS
FROM year_month_order
ORDER BY YEAR_MONTH DESC

/*Insight: New customer has increased steadily, depend on inscreasing new customer total orders also insceased, but in 2026/05,it seems tending to decrease*/

### b. Monthly revenue




In [ ]:
%%sql

WITH REVENUE_ORDER AS
(
  SELECT
EXTRACT(YEAR FROM oi.CREATED_AT) AS YEAR,
EXTRACT(MONTH FROM oi.CREATED_AT) AS MONTH,
COUNT(DISTINCT oi.USER_ID) AS TOTAL_USERS,
COUNT(DISTINCT oi.ORDER_ID) AS TOTAL_ORDERS,
ROUND(SUM(oi.SALE_PRICE * o.NUM_OF_ITEM),2) AS REVENUE
FROM order_items AS oi
JOIN orders AS o ON A.ORDER_ID= o.ORDER_ID
WHERE oi.STATUS IN ('Complete')
GROUP BY YEAR, MONTH)

SELECT
CAST(YEAR AS STRING) ||'-'|| CAST(MONTH AS STRING) AS YEAR_MONTH,
TOTAL_USERS AS DISTINCT_USER,
TOTAL_ORDERS,
REVENUE
FROM REVENUE_ORDER
ORDER BY REVENUE DESC

## 2. Customer profile
### a. Customer order by country

In [ ]:
%%sql

select
u.country,
count(distinct o.order_id) AS total_orders,
count(distinct u.id) AS total_customers
from orders as o
left join public.users as u on u.id = o.user_id
where o.status in ('Complete')
group by u.country
order by count(distinct o.order_id) desc;
--- China is a country has the most orders and customer

###b. Customers by Gender

In [ ]:
%%sql

select
o.gender as gender,
round(sum(oi.sale_price * o.num_of_item)::numeric,2) AS revenue,
sum(num_of_item)AS quantity
from orders as o
left join public.order_items as oi on oi.order_id = o.order_id
where o.status in ('Complete')
group by o.gender
order by revenue desc;


###c. Customers by Age Group

In [ ]:
%%sql

select
case
	when u.age < 12 then 'Kid'
	when u.age between 12 and 20 then 'Teenager'
	when u.age between 20 and 30 then 'Young Adult'
	when u.age between 30 and 50 then 'Adult'
	else 'Elderly'
end As age_group,
count(distinct o.order_id) as total_customer
from orders as o
left join public.users as u on u.id = o.order_id
where o.status in ('Complete')
group by age_group
order by count(distinct o.order_id) desc;